# AI-Powered Business Complaint Intelligence Platform
## DistilBERT Sentiment Analysis Model (Complaint Domain)

**FYP Project:** Business Complaint Intelligence Platform  
**Team:** Salman Ali (CS-296), Ayesha Ali (CS-281), Kashaf Shabbir (CS-285)  
**Supervisor:** Prof. Ayesha Khan  
**Institution:** National University of Modern Languages, Lahore

In [1]:
# ============================================================
# BCIP MODEL 1 — SENTIMENT ANALYSIS
# DistilBERT Fine-Tuning on Pakistani Complaint Data
# Google Colab
# ============================================================

%pip install -q -U \
    transformers \
    datasets \
    accelerate \
    evaluate \
    scikit-learn \
    pandas \
    numpy \
    matplotlib \
    seaborn \
    openpyxl \
    pyarrow

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement seaborn (from versions: none)
ERROR: No matching distribution found for seaborn


### **Imports + reproducibility**

In [29]:
import os
import re
import json
import random
import shutil
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Colab GPU is not enabled.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


### **Upload your actual complaint CSV**

In [30]:
# ============================================================
# UPLOAD ACTUAL FYP COMPLAINT DATASET
# ============================================================

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No dataset uploaded.")

uploaded_name = next(iter(uploaded.keys()))
DATA_PATH = Path(uploaded_name)

print(f"Dataset uploaded: {DATA_PATH}")
print(f"File size: {DATA_PATH.stat().st_size / 1024:.2f} KB")

Saving Roman Urdu DataSet.csv to Roman Urdu DataSet.csv
Dataset uploaded: Roman Urdu DataSet.csv
File size: 1590.18 KB


### **Robust CSV loader**

In [31]:
# ============================================================
# ROBUST DATASET LOADER
# Handles:
# - CSV
# - Excel
# - Parquet
# - Encoding issues
# - Bad separators
# ============================================================

def load_dataset_file(path: Path) -> pd.DataFrame:

    suffix = path.suffix.lower()

    if suffix == ".csv":

        encodings = [
            "utf-8",
            "utf-8-sig",
            "cp1252",
            "latin1",
        ]

        errors = []

        for encoding in encodings:
            try:
                df = pd.read_csv(
                    path,
                    encoding=encoding,
                    sep=None,
                    engine="python",
                )

                if df.shape[1] >= 1:
                    print(f"Loaded CSV using encoding: {encoding}")
                    return df

            except Exception as e:
                errors.append(f"{encoding}: {e}")

        raise RuntimeError(
            "Unable to read CSV.\n" + "\n".join(errors)
        )

    elif suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    elif suffix == ".parquet":
        return pd.read_parquet(path)

    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. "
            "Use CSV, XLSX, XLS, or Parquet."
        )


complaints_df = load_dataset_file(DATA_PATH)

print("\nDataset loaded successfully")
print("=" * 70)

print("Rows:", len(complaints_df))
print("Columns:", complaints_df.columns.tolist())

display(complaints_df.head())

Loaded CSV using encoding: utf-8

Dataset loaded successfully
Rows: 20228
Columns: ['Sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi Aj aur aj bi sirf Aus say bus', 'Positive', 'Unnamed: 2']


,Sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi Aj aur aj bi sirf Aus say bus,Positive,Unnamed: 2
0,sahi bt h,Positive,NaN
1,"Kya bt hai,",Positive,NaN
2,Wah je wah,Positive,NaN
3,Are wha kaya bat hai,Positive,NaN
4,Wah kya baat likhi,Positive,NaN


### **Dataset audit**

In [32]:
# ============================================================
# DATASET QUALITY AUDIT
# ============================================================

print("\nDATASET AUDIT")
print("=" * 70)

print("\nShape:")
print(complaints_df.shape)

print("\nData types:")
display(
    complaints_df.dtypes.to_frame("dtype")
)

print("\nMissing values:")
missing = (
    complaints_df.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing[missing > 0]
    .to_frame("missing_count")
)

print("\nDuplicate rows:")
print(complaints_df.duplicated().sum())

print("\nSample:")
display(complaints_df.head(10))


DATASET AUDIT

Shape:
(20228, 3)

Data types:


,dtype
Sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi Aj aur aj bi sirf Aus say bus,object
Positive,object
Unnamed: 2,object



Missing values:


,missing_count
Unnamed: 2,20221
Sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi Aj aur aj bi sirf Aus say bus,1



Duplicate rows:
543

Sample:


,Sai kha ya her kisi kay bus ki bat nhi hai lakin main ki hal kal bi Aj aur aj bi sirf Aus say bus,Positive,Unnamed: 2
0,sahi bt h,Positive,NaN
1,"Kya bt hai,",Positive,NaN
2,Wah je wah,Positive,NaN
3,Are wha kaya bat hai,Positive,NaN
4,Wah kya baat likhi,Positive,NaN
5,Wha Itni sari khubiya,Positive,NaN
6,Itni khubiya,Positive,NaN
7,Ya allah rehm farma hm sab pe or zalimo ko hid...,Positive,NaN
8,Please Everyone AllAh S.w.T ka naam hAmesha Ba...,Positive,NaN
9,Ya mere rab tu bra kreem hy is mulk k halat PR...,Positive,NaN


### **Automatically find the complaint-text column**

In [36]:
# ============================================================
# AUTOMATIC TEXT COLUMN DETECTION
# ============================================================

TEXT_CANDIDATES = [
    "complaint_text",
    "complaint",
    "complaint_text_en",
    "text",
    "message",
    "description",
    "feedback",
    "review",
    "customer_complaint",
    "issue",
    "remarks",
]


def normalize_column_name(column):
    return (
        str(column)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


# ------------------------------------------------------------
# Normalize column names
# ------------------------------------------------------------

original_columns = complaints_df.columns.tolist()

complaints_df.columns = [
    normalize_column_name(c)
    for c in complaints_df.columns
]

print("Original columns:")
print(original_columns)

print("\nNormalized columns:")
print(complaints_df.columns.tolist())


# ------------------------------------------------------------
# Find complaint text column
# ------------------------------------------------------------

def find_text_column(df):

    # --------------------------------------------------------
    # 1. Exact candidate match
    # --------------------------------------------------------

    for candidate in TEXT_CANDIDATES:
        if candidate in df.columns:
            return candidate


    # --------------------------------------------------------
    # 2. Fuzzy name-based detection
    # --------------------------------------------------------

    text_like_columns = []

    keywords = [
        "complaint",
        "message",
        "text",
        "description",
        "feedback",
        "issue",
        "comment",
        "review",
        "remarks",
    ]

    for col in df.columns:

        name = str(col).lower()

        if any(keyword in name for keyword in keywords):
            text_like_columns.append(col)


    if len(text_like_columns) == 1:
        return text_like_columns[0]


    if len(text_like_columns) > 1:

        scores = {}

        for col in text_like_columns:

            values = (
                df[col]
                .dropna()
                .astype(str)
                .str.strip()
            )

            if len(values) > 0:
                scores[col] = values.str.len().mean()
            else:
                scores[col] = 0

        return max(scores, key=scores.get)


    # --------------------------------------------------------
    # 3. CONTENT-BASED detection
    #
    # If the column names are useless, determine which column
    # actually contains long text.
    # --------------------------------------------------------

    content_scores = {}

    for col in df.columns:

        values = (
            df[col]
            .dropna()
            .astype(str)
            .str.strip()
        )

        if len(values) == 0:
            content_scores[col] = 0
            continue

        # Average text length
        avg_length = values.str.len().mean()

        # Percentage of non-empty string values
        non_empty_ratio = (
            values.str.len().gt(0).mean()
        )

        # Combined score
        content_scores[col] = (
            avg_length * non_empty_ratio
        )


    # --------------------------------------------------------
    # Select column with strongest text characteristics
    # --------------------------------------------------------

    if content_scores:

        best_column = max(
            content_scores,
            key=content_scores.get
        )

        best_score = content_scores[best_column]

        if best_score > 10:
            return best_column


    return None


# ------------------------------------------------------------
# Detect column
# ------------------------------------------------------------

TEXT_COLUMN = find_text_column(complaints_df)


# ------------------------------------------------------------
# Safety fallback
# ------------------------------------------------------------

if TEXT_COLUMN is None:

    print(
        "\nAutomatic detection failed."
    )

    print(
        "\nAvailable columns:"
    )

    for col in complaints_df.columns:
        print(" -", col)

    raise ValueError(
        "\nCould not automatically identify the complaint-text column.\n"
        "Please set TEXT_COLUMN manually."
    )


print(
    "\nDetected complaint text column:",
    TEXT_COLUMN
)


# ------------------------------------------------------------
# Optional: show sample complaints
# ------------------------------------------------------------

print("\nSample complaint texts:")

print(
    complaints_df[TEXT_COLUMN]
    .dropna()
    .astype(str)
    .head(5)
    .to_string(index=False)
)

Original columns:
['sai_kha_ya_her_kisi_kay_bus_ki_bat_nhi_hai_lakin_main_ki_hal_kal_bi_aj_aur_aj_bi_sirf_aus_say_bus', 'positive', 'unnamed:_2']

Normalized columns:
['sai_kha_ya_her_kisi_kay_bus_ki_bat_nhi_hai_lakin_main_ki_hal_kal_bi_aj_aur_aj_bi_sirf_aus_say_bus', 'positive', 'unnamed:_2']

Detected complaint text column: sai_kha_ya_her_kisi_kay_bus_ki_bat_nhi_hai_lakin_main_ki_hal_kal_bi_aj_aur_aj_bi_sirf_aus_say_bus

Sample complaint texts:
           sahi bt h
         Kya bt hai,
          Wah je wah
Are wha kaya bat hai
  Wah kya baat likhi


### **Clean complaint text**

In [37]:
# ============================================================
# TEXT CLEANING
# ============================================================

def clean_complaint_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize Unicode whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text


complaints_df[TEXT_COLUMN] = (
    complaints_df[TEXT_COLUMN]
    .apply(clean_complaint_text)
)

# Remove empty complaints
before = len(complaints_df)

complaints_df = complaints_df[
    complaints_df[TEXT_COLUMN].str.len() > 2
].copy()

after = len(complaints_df)

print("Removed empty/invalid complaints:", before - after)

# Remove exact duplicate complaint texts
before = len(complaints_df)

complaints_df = (
    complaints_df
    .drop_duplicates(
        subset=[TEXT_COLUMN],
        keep="first"
    )
    .reset_index(drop=True)
)

after = len(complaints_df)

print("Removed duplicate complaint texts:", before - after)
print("Final dataset size:", len(complaints_df))

Removed empty/invalid complaints: 192
Removed duplicate complaint texts: 492
Final dataset size: 19544


### **Find an existing sentiment label**

In [38]:
# ============================================================
# DETECT EXISTING SENTIMENT LABEL
# ============================================================

LABEL_CANDIDATES = [
    "sentiment_label",
    "sentiment",
    "sentiment_class",
    "sentiment_category",
    "label",
    "polarity",
    "emotion",
]

LABEL_COLUMN = None

for candidate in LABEL_CANDIDATES:
    if candidate in complaints_df.columns:
        LABEL_COLUMN = candidate
        break

print("Detected sentiment column:", LABEL_COLUMN)

if LABEL_COLUMN is not None:
    print("\nExisting label distribution:")
    display(
        complaints_df[LABEL_COLUMN]
        .value_counts(dropna=False)
        .to_frame("count")
    )
else:
    print(
        "\nNo sentiment label exists in the dataset."
        "\nWe will create weak-supervision labels in the next cell."
    )

Detected sentiment column: None

No sentiment label exists in the dataset.
We will create weak-supervision labels in the next cell.


### **Sentiment label preparation**

In [39]:
# ============================================================
# SENTIMENT LABEL NORMALIZATION
#
# Preferred:
#     Human-labeled sentiment
#
# Fallback:
#     Weak supervision
#
# IMPORTANT:
# Weak labels are NOT independent ground truth.
# ============================================================

SENTIMENT_RUBRIC = {

    "negative": [
        "angry",
        "frustrated",
        "disappointed",
        "upset",
        "terrible",
        "bad",
        "horrible",
        "worst",
        "waste",
        "damaged",
        "broken",
        "not received",
        "delayed",
        "charged twice",
        "scam",
        "fraud",
        "useless",
        "pathetic",
        "disgusting",
        "hate",
        "late",
        "missing",
        "failed",
        "unacceptable",
        "ignored",
        "poor",
        "awful",
        "never arrived",
        "not working",
        "wrong item",
        "refund",
    ],

    "positive": [
        "satisfied",
        "happy",
        "good",
        "excellent",
        "perfect",
        "great",
        "wonderful",
        "amazing",
        "thankful",
        "appreciate",
        "resolved",
        "quickly",
        "helpful",
        "professional",
        "recommend",
        "smooth",
        "on time",
        "fixed",
        "thank you",
    ],
}


def weak_label_sentiment(text):

    text = str(text).lower()

    negative_score = 0
    positive_score = 0

    for term in SENTIMENT_RUBRIC["negative"]:
        if term in text:
            negative_score += 1

    for term in SENTIMENT_RUBRIC["positive"]:
        if term in text:
            positive_score += 1

    # Tie/default -> negative because the dataset is complaint-oriented
    if positive_score > negative_score:
        return 1

    return 0


def normalize_sentiment_label(value):

    if pd.isna(value):
        return np.nan

    # Numeric labels
    if isinstance(value, (int, float, np.integer, np.floating)):

        value = int(value)

        if value in [0, 1]:
            return value

    value = str(value).strip().lower()

    positive_values = {
        "positive",
        "pos",
        "1",
        "true",
        "good",
        "satisfied",
        "happy",
    }

    negative_values = {
        "negative",
        "neg",
        "0",
        "false",
        "bad",
        "unsatisfied",
        "angry",
    }

    if value in positive_values:
        return 1

    if value in negative_values:
        return 0

    return np.nan


if LABEL_COLUMN is not None:

    complaints_df["sentiment_label"] = (
        complaints_df[LABEL_COLUMN]
        .apply(normalize_sentiment_label)
    )

    # Remove rows with invalid labels
    before = len(complaints_df)

    complaints_df = complaints_df[
        complaints_df["sentiment_label"]
        .notna()
    ].copy()

    after = len(complaints_df)

    print("Removed rows with invalid sentiment labels:", before - after)

    LABEL_SOURCE = f"existing dataset labels: {LABEL_COLUMN}"

else:

    print(
        "WARNING: No human sentiment labels detected."
        "\nGenerating weak-supervision labels."
    )

    complaints_df["sentiment_label"] = (
        complaints_df[TEXT_COLUMN]
        .apply(weak_label_sentiment)
    )

    LABEL_SOURCE = "keyword-based weak supervision"


complaints_df["sentiment_label"] = (
    complaints_df["sentiment_label"]
    .astype(int)
)

print("\nLabel source:", LABEL_SOURCE)

print("\nSentiment distribution:")
display(
    complaints_df["sentiment_label"]
    .value_counts()
    .sort_index()
    .rename(index={
        0: "NEGATIVE",
        1: "POSITIVE"
    })
    .to_frame("count")
)

Generating weak-supervision labels.

Label source: keyword-based weak supervision

Sentiment distribution:


,count
sentiment_label,
NEGATIVE,19273
POSITIVE,271


### **Stratified train/validation/test split**

In [40]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
#
# 70% train
# 15% validation
# 15% test
#
# Stratification preserves sentiment distribution.
# ============================================================

train_df, temp_df = train_test_split(
    complaints_df,
    test_size=0.30,
    stratify=complaints_df["sentiment_label"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["sentiment_label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Dataset split")
print("=" * 70)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["sentiment_label"].value_counts(normalize=True))

print("\nValidation distribution:")
print(val_df["sentiment_label"].value_counts(normalize=True))

print("\nTest distribution:")
print(test_df["sentiment_label"].value_counts(normalize=True))

Dataset split
Train: 13680
Validation: 2932
Test: 2932

Train distribution:
sentiment_label
0    0.986111
1    0.013889
Name: proportion, dtype: float64

Validation distribution:
sentiment_label
0    0.986016
1    0.013984
Name: proportion, dtype: float64

Test distribution:
sentiment_label
0    0.986357
1    0.013643
Name: proportion, dtype: float64


### **Convert to Hugging Face datasets**

In [41]:
# ============================================================
# HUGGING FACE DATASETS
# ============================================================

def to_hf_dataset(df):

    output = df[
        [TEXT_COLUMN, "sentiment_label"]
    ].copy()

    output = output.rename(
        columns={
            TEXT_COLUMN: "text",
            "sentiment_label": "labels",
        }
    )

    return Dataset.from_pandas(
        output,
        preserve_index=False
    )


dataset_train = to_hf_dataset(train_df)
dataset_val = to_hf_dataset(val_df)
dataset_test = to_hf_dataset(test_df)

print(dataset_train)
print(dataset_val)
print(dataset_test)

Dataset({
    features: ['text', 'labels'],
    num_rows: 13680
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 2932
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 2932
})


### **DistilBERT tokenizer**

In [42]:
# ============================================================
# DISTILBERT TOKENIZER
# ============================================================

BASE_MODEL = "distilbert/distilbert-base-uncased"

MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)


def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_train = dataset_train.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train"
)

tokenized_val = dataset_val.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing validation"
)

tokenized_test = dataset_test.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing test"
)

print(tokenized_train)

Tokenizing train:   0%|          | 0/13680 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/2932 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/2932 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 13680
})


### **DistilBERT model**

In [43]:
# ============================================================
# DISTILBERT SEQUENCE CLASSIFICATION MODEL
# ============================================================

id2label = {
    0: "NEGATIVE",
    1: "POSITIVE",
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1,
}


model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 66955010


### **Dynamic padding**

In [44]:
# ============================================================
# DYNAMIC PADDING
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

### **Class-weighted loss**

In [45]:
# ============================================================
# CLASS WEIGHTING
#
# Prevents the model from simply predicting the majority class.
# ============================================================

train_counts = (
    train_df["sentiment_label"]
    .value_counts()
    .sort_index()
)

num_samples = len(train_df)
num_classes = 2

class_weights = torch.tensor(
    [
        num_samples / (
            num_classes * train_counts.loc[0]
        ),

        num_samples / (
            num_classes * train_counts.loc[1]
        ),
    ],
    dtype=torch.float32,
)

print("Class weights:")
print(
    "NEGATIVE:",
    float(class_weights[0])
)

print(
    "POSITIVE:",
    float(class_weights[1])
)

Class weights:
NEGATIVE: 0.5070422291755676
POSITIVE: 36.0


### **Evaluation metrics**

In [46]:
# ============================================================
# EVALUATION METRICS
#
# Accuracy alone is insufficient.
# We report:
# - Accuracy
# - Macro Precision
# - Macro Recall
# - Macro F1
# - Negative F1
# - Positive F1
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        )
    )

    _, _, f1_per_class, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average=None,
            labels=[0, 1],
            zero_division=0,
        )
    )

    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions
            )
        ),

        "precision_macro": float(
            precision_macro
        ),

        "recall_macro": float(
            recall_macro
        ),

        "f1_macro": float(
            f1_macro
        ),

        "f1_negative": float(
            f1_per_class[0]
        ),

        "f1_positive": float(
            f1_per_class[1]
        ),
    }

### **Weighted Trainer**

In [48]:
# ============================================================
# CUSTOM TRAINER WITH WEIGHTED CROSS-ENTROPY
# ============================================================

class WeightedLossTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        loss_function = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(
                outputs.logits.device
            )
        )

        loss = loss_function(
            outputs.logits,
            labels
        )

        if return_outputs:
            return loss, outputs

        return loss

### **Training configuration**

In [50]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

OUTPUT_DIR = "/content/bcip_sentiment_checkpoints"
FINAL_MODEL_DIR = "/content/bcip_sentiment_model"

NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,

    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",

    greater_is_better=True,

    logging_strategy="epoch",

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    seed=SEED,

    report_to="none",

    push_to_hub=False,
)

### **Create Trainer**

In [51]:
# ============================================================
# TRAINER
# ============================================================

trainer = WeightedLossTrainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_val,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

print("Trainer initialized successfully.")

Trainer initialized successfully.


### **Train**

In [52]:
# ============================================================
# TRAIN DISTILBERT
# ============================================================

print("=" * 70)
print("STARTING DISTILBERT TRAINING")
print("=" * 70)

training_start = time.perf_counter()

train_result = trainer.train()

training_time = (
    time.perf_counter()
    - training_start
)

print("\nTraining complete.")
print(
    f"Training time: "
    f"{training_time / 60:.2f} minutes"
)

print("\nTraining result:")
print(train_result)

STARTING DISTILBERT TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Negative,F1 Positive
1,0.298763,0.311204,0.996248,0.956952,0.901920,0.927621,0.998099,0.857143
2,0.071824,0.122750,0.997271,0.941341,0.962550,0.951689,0.998616,0.904762
3,0.038858,0.113015,0.998295,0.974481,0.963069,0.968704,0.999135,0.938272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
Training time: 2.95 minutes

Training result:
TrainOutput(global_step=2565, training_loss=0.13648142675210162, metrics={'train_runtime': 176.1361, 'train_samples_per_second': 233.002, 'train_steps_per_second': 14.563, 'total_flos': 828393156707712.0, 'train_loss': 0.13648142675210162, 'epoch': 3.0})


### **Validation evaluation**

In [53]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

validation_results = trainer.evaluate()

print("=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

for key, value in validation_results.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

    else:
        print(f"{key}: {value}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Negative,F1 Positive
0.038858,0.113015,3,0.998295,0.974481,0.963069,0.968704,0.999135,0.938272


VALIDATION RESULTS
eval_loss: 0.1130
eval_accuracy: 0.9983
eval_precision_macro: 0.9745
eval_recall_macro: 0.9631
eval_f1_macro: 0.9687
eval_f1_negative: 0.9991
eval_f1_positive: 0.9383


### **Final untouched test evaluation**

In [54]:
# ============================================================
# FINAL TEST EVALUATION
#
# The test set was never used for model selection.
# ============================================================

test_results = trainer.evaluate(
    eval_dataset=tokenized_test
)

print("=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

for key, value in test_results.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

    else:
        print(f"{key}: {value}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Negative,F1 Positive
0.038858,0.031022,3,0.998977,0.975437,0.987154,0.981222,0.999481,0.962963


FINAL TEST RESULTS
eval_loss: 0.0310
eval_accuracy: 0.9990
eval_precision_macro: 0.9754
eval_recall_macro: 0.9872
eval_f1_macro: 0.9812
eval_f1_negative: 0.9995
eval_f1_positive: 0.9630


### **Classification report + confusion matrix**

In [55]:
# ============================================================
# DETAILED TEST REPORT
# ============================================================

prediction_output = trainer.predict(
    tokenized_test
)

test_predictions = np.argmax(
    prediction_output.predictions,
    axis=-1
)

test_labels = np.asarray(
    prediction_output.label_ids
)


print(
    classification_report(
        test_labels,
        test_predictions,
        labels=[0, 1],
        target_names=[
            "NEGATIVE",
            "POSITIVE",
        ],
        digits=4,
        zero_division=0,
    )
)


cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=[0, 1],
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual NEGATIVE",
        "Actual POSITIVE",
    ],
    columns=[
        "Predicted NEGATIVE",
        "Predicted POSITIVE",
    ],
)

display(cm_df)

              precision    recall  f1-score   support

    NEGATIVE     0.9997    0.9993    0.9995      2892
    POSITIVE     0.9512    0.9750    0.9630        40

    accuracy                         0.9990      2932
   macro avg     0.9754    0.9872    0.9812      2932
weighted avg     0.9990    0.9990    0.9990      2932



,Predicted NEGATIVE,Predicted POSITIVE
Actual NEGATIVE,2890,2
Actual POSITIVE,1,39


### **Save production model**

In [56]:
# ============================================================
# SAVE FINAL MODEL + TOKENIZER
# ============================================================

MODEL_PATH = Path(FINAL_MODEL_DIR)

MODEL_PATH.mkdir(
    parents=True,
    exist_ok=True
)

trainer.save_model(
    str(MODEL_PATH)
)

tokenizer.save_pretrained(
    str(MODEL_PATH)
)

print("Model saved to:")
print(MODEL_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to:
/content/bcip_sentiment_model


### **Reload exactly as production inference will**

In [57]:
# ============================================================
# PRODUCTION RELOAD TEST
# ============================================================

loaded_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

loaded_model = (
    AutoModelForSequenceClassification
    .from_pretrained(MODEL_PATH)
)

device = (
    0
    if torch.cuda.is_available()
    else -1
)

sentiment_classifier = pipeline(
    "text-classification",

    model=loaded_model,

    tokenizer=loaded_tokenizer,

    device=device,
)

print("Production reload successful.")
print(
    "Labels:",
    loaded_model.config.id2label
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Production reload successful.
Labels: {0: 'NEGATIVE', 1: 'POSITIVE'}


### **Pakistani complaint inference test**

In [58]:
# ============================================================
# BCIP REAL-WORLD INFERENCE TEST
# ============================================================

test_complaints = [

    "Mera order 6 din se nahi aya aur tracking 3 din se same hai.",

    "Bohat acha experience tha, order time par mil gaya. Thank you!",

    "Mujhe same order ke liye do dafa charge kiya gaya hai.",

    "Delivery late thi lekin customer support ne issue jaldi solve kar diya.",

    "Product damaged aya hai aur seller meri complaint ignore kar raha hai.",

    "Everything arrived perfectly and I am very satisfied.",
]


results = sentiment_classifier(
    test_complaints,
    truncation=True,
    max_length=MAX_LENGTH,
)


for text, result in zip(
    test_complaints,
    results
):

    print("=" * 90)

    print("Complaint:")
    print(text)

    print(
        "\nSentiment:",
        result["label"]
    )

    print(
        "Confidence:",
        f"{result['score'] * 100:.2f}%"
    )

Complaint:
Mera order 6 din se nahi aya aur tracking 3 din se same hai.

Sentiment: NEGATIVE
Confidence: 99.98%
Complaint:
Bohat acha experience tha, order time par mil gaya. Thank you!

Sentiment: POSITIVE
Confidence: 99.96%
Complaint:
Mujhe same order ke liye do dafa charge kiya gaya hai.

Sentiment: NEGATIVE
Confidence: 99.99%
Complaint:
Delivery late thi lekin customer support ne issue jaldi solve kar diya.

Sentiment: NEGATIVE
Confidence: 99.97%
Complaint:
Product damaged aya hai aur seller meri complaint ignore kar raha hai.

Sentiment: NEGATIVE
Confidence: 99.98%
Complaint:
Everything arrived perfectly and I am very satisfied.

Sentiment: POSITIVE
Confidence: 75.73%


### **Production confidence threshold**

In [59]:
# ============================================================
# CONFIDENCE-AWARE INFERENCE
#
# Low-confidence predictions should NOT be blindly trusted.
# ============================================================

CONFIDENCE_THRESHOLD = 0.85


def predict_sentiment(text):

    result = sentiment_classifier(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
    )[0]

    label = result["label"]
    confidence = float(result["score"])

    requires_review = (
        confidence < CONFIDENCE_THRESHOLD
    )

    return {
        "sentiment": label,
        "confidence": round(confidence, 4),
        "requires_human_review": requires_review,
    }


examples = [
    "My parcel is extremely late and nobody is responding.",
    "Thank you, the issue was resolved quickly.",
    "Mera parcel abhi tak nahi aya.",
]

for example in examples:

    print("\nComplaint:")
    print(example)

    print(
        json.dumps(
            predict_sentiment(example),
            indent=2,
            ensure_ascii=False,
        )
    )


Complaint:
My parcel is extremely late and nobody is responding.
{
  "sentiment": "NEGATIVE",
  "confidence": 0.999,
  "requires_human_review": false
}

Complaint:
Thank you, the issue was resolved quickly.
{
  "sentiment": "POSITIVE",
  "confidence": 0.9996,
  "requires_human_review": false
}

Complaint:
Mera parcel abhi tak nahi aya.
{
  "sentiment": "NEGATIVE",
  "confidence": 0.9999,
  "requires_human_review": false
}


### **CPU/GPU inference benchmark**

In [60]:
# ============================================================
# INFERENCE BENCHMARK
# ============================================================

benchmark_complaints = [
    "Mera order 5 din se nahi aya.",
    "Product quality is very bad.",
    "Payment failed multiple times.",
    "Customer service ignored me.",
    "Order delivered on time. Very satisfied!",
    "The issue was resolved quickly.",
] * 20

# Warm-up
_ = sentiment_classifier(
    benchmark_complaints[:4],
    truncation=True,
    max_length=MAX_LENGTH,
)

start = time.perf_counter()

benchmark_results = sentiment_classifier(
    benchmark_complaints,
    truncation=True,
    max_length=MAX_LENGTH,
    batch_size=32,
)

elapsed = time.perf_counter() - start

average_latency_ms = (
    elapsed
    / len(benchmark_complaints)
    * 1000
)

throughput_rps = (
    len(benchmark_complaints)
    / elapsed
)

print("=" * 70)
print("INFERENCE BENCHMARK")
print("=" * 70)

print(
    "Device:",
    "GPU" if torch.cuda.is_available() else "CPU"
)

print(
    "Requests:",
    len(benchmark_complaints)
)

print(
    f"Total time: {elapsed:.3f}s"
)

print(
    f"Average latency: "
    f"{average_latency_ms:.2f} ms/request"
)

print(
    f"Throughput: "
    f"{throughput_rps:.2f} requests/sec"
)

print(
    "\nIMPORTANT:"
    "\nDo NOT claim <100ms unless this benchmark actually passes."
)

INFERENCE BENCHMARK
Device: GPU
Requests: 120
Total time: 0.105s
Average latency: 0.88 ms/request
Throughput: 1139.80 requests/sec

IMPORTANT:
Do NOT claim <100ms unless this benchmark actually passes.


### **Save metadata for the FYP**

In [61]:
# ============================================================
# MODEL VERSION METADATA
# ============================================================

metadata = {

    "model_name":
        "bcip-distilbert-complaint-sentiment-v1",

    "base_model":
        BASE_MODEL,

    "task":
        "binary complaint sentiment classification",

    "labels": {
        "0": "NEGATIVE",
        "1": "POSITIVE",
    },

    "text_column":
        TEXT_COLUMN,

    "label_source":
        LABEL_SOURCE,

    "dataset": {

        "total_rows":
            int(len(complaints_df)),

        "train_rows":
            int(len(train_df)),

        "validation_rows":
            int(len(val_df)),

        "test_rows":
            int(len(test_df)),
    },

    "class_distribution": {

        "negative":
            int(
                (train_df["sentiment_label"] == 0)
                .sum()
            ),

        "positive":
            int(
                (train_df["sentiment_label"] == 1)
                .sum()
            ),
    },

    "hyperparameters": {

        "learning_rate":
            LEARNING_RATE,

        "epochs":
            NUM_EPOCHS,

        "train_batch_size":
            TRAIN_BATCH_SIZE,

        "eval_batch_size":
            EVAL_BATCH_SIZE,

        "max_length":
            MAX_LENGTH,

        "weight_decay":
            WEIGHT_DECAY,

        "seed":
            SEED,
    },

    "validation_metrics": {

        "accuracy":
            float(
                validation_results["eval_accuracy"]
            ),

        "precision_macro":
            float(
                validation_results["eval_precision_macro"]
            ),

        "recall_macro":
            float(
                validation_results["eval_recall_macro"]
            ),

        "f1_macro":
            float(
                validation_results["eval_f1_macro"]
            ),

        "f1_negative":
            float(
                validation_results["eval_f1_negative"]
            ),

        "f1_positive":
            float(
                validation_results["eval_f1_positive"]
            ),
    },

    "test_metrics": {

        "accuracy":
            float(
                test_results["eval_accuracy"]
            ),

        "precision_macro":
            float(
                test_results["eval_precision_macro"]
            ),

        "recall_macro":
            float(
                test_results["eval_recall_macro"]
            ),

        "f1_macro":
            float(
                test_results["eval_f1_macro"]
            ),

        "f1_negative":
            float(
                test_results["eval_f1_negative"]
            ),

        "f1_positive":
            float(
                test_results["eval_f1_positive"]
            ),
    },

    "inference": {

        "device":
            "cuda"
            if torch.cuda.is_available()
            else "cpu",

        "average_latency_ms":
            round(
                average_latency_ms,
                3
            ),

        "throughput_requests_per_second":
            round(
                throughput_rps,
                3
            ),

        "confidence_threshold":
            CONFIDENCE_THRESHOLD,
    },

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "research_warning":
        (
            "If label_source is keyword-based weak "
            "supervision, evaluation metrics must not "
            "be presented as independent human-ground-truth "
            "performance."
        ),
}


metadata_path = MODEL_PATH / "metadata.json"

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


print(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "model_name": "bcip-distilbert-complaint-sentiment-v1",
  "base_model": "distilbert/distilbert-base-uncased",
  "task": "binary complaint sentiment classification",
  "labels": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "text_column": "sai_kha_ya_her_kisi_kay_bus_ki_bat_nhi_hai_lakin_main_ki_hal_kal_bi_aj_aur_aj_bi_sirf_aus_say_bus",
  "label_source": "keyword-based weak supervision",
  "dataset": {
    "total_rows": 19544,
    "train_rows": 13680,
    "validation_rows": 2932,
    "test_rows": 2932
  },
  "class_distribution": {
    "negative": 13490,
    "positive": 190
  },
  "hyperparameters": {
    "learning_rate": 2e-05,
    "epochs": 3,
    "train_batch_size": 16,
    "eval_batch_size": 32,
    "max_length": 128,
    "weight_decay": 0.01,
    "seed": 42
  },
  "validation_metrics": {
    "accuracy": 0.9982946793997272,
    "precision_macro": 0.9744813278008299,
    "recall_macro": 0.9630687330740482,
    "f1_macro": 0.968703500895558,
    "f1_negative": 0.9991353968528

In [62]:
def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        predicted_class = torch.argmax(
            probabilities,
            dim=-1
        ).item()

    confidence = probabilities[0][predicted_class].item()

    label = model.config.id2label[predicted_class]

    return {
        "sentiment": label,
        "confidence": confidence
    }


# ============================================================
# TEST
# ============================================================

complaint = "The service was excellent and I am very satisfied."

result = predict_sentiment(complaint)

print("Complaint:")
print(complaint)

print("\nPrediction:")
print(result["sentiment"])

print("Confidence:")
print(f"{result['confidence']:.2%}")

Complaint:
The service was excellent and I am very satisfied.

Prediction:
POSITIVE
Confidence:
99.95%


In [63]:
test_complaints = [
    "The service was excellent and very fast.",
    "I am extremely disappointed with this service.",
    "The staff was helpful and professional.",
    "My complaint has not been resolved for three days.",
    "Everything worked perfectly.",
    "This is the worst service I have ever experienced.",
    "The response was okay, nothing special.",
    "I am happy with the quick response."
]


print("=" * 70)
print("COMPLAINT SENTIMENT TESTING")
print("=" * 70)


for complaint in test_complaints:

    result = predict_sentiment(complaint)

    print("\nComplaint:")
    print(complaint)

    print(
        f"Prediction: {result['sentiment']}"
    )

    print(
        f"Confidence: {result['confidence']:.2%}"
    )

    print("-" * 70)

COMPLAINT SENTIMENT TESTING

Complaint:
The service was excellent and very fast.
Prediction: POSITIVE
Confidence: 99.95%
----------------------------------------------------------------------

Complaint:
I am extremely disappointed with this service.
Prediction: NEGATIVE
Confidence: 99.42%
----------------------------------------------------------------------

Complaint:
The staff was helpful and professional.
Prediction: POSITIVE
Confidence: 99.53%
----------------------------------------------------------------------

Complaint:
My complaint has not been resolved for three days.
Prediction: NEGATIVE
Confidence: 99.80%
----------------------------------------------------------------------

Complaint:
Everything worked perfectly.
Prediction: NEGATIVE
Confidence: 99.69%
----------------------------------------------------------------------

Complaint:
This is the worst service I have ever experienced.
Prediction: NEGATIVE
Confidence: 99.87%
----------------------------------------------

In [64]:
roman_urdu_complaints = [
    "Mujhe service bohat achi lagi hai.",
    "Meri complaint ka koi solution nahi diya gaya.",
    "Staff ka rawayya bohat acha tha.",
    "Ye service bilkul bakwas hai.",
    "Main is service se bohat khush hun.",
    "Teen din ho gaye hain lekin meri complaint solve nahi hui.",
    "Bohat acha response mila.",
    "Mujhe bohat zyada pareshani hui hai."
]


print("=" * 70)
print("ROMAN URDU SENTIMENT TESTING")
print("=" * 70)


for complaint in roman_urdu_complaints:

    result = predict_sentiment(complaint)

    print("\nComplaint:")
    print(complaint)

    print(
        f"Prediction: {result['sentiment']}"
    )

    print(
        f"Confidence: {result['confidence']:.2%}"
    )

    print("-" * 70)

ROMAN URDU SENTIMENT TESTING

Complaint:
Mujhe service bohat achi lagi hai.
Prediction: NEGATIVE
Confidence: 99.99%
----------------------------------------------------------------------

Complaint:
Meri complaint ka koi solution nahi diya gaya.
Prediction: NEGATIVE
Confidence: 99.99%
----------------------------------------------------------------------

Complaint:
Staff ka rawayya bohat acha tha.
Prediction: NEGATIVE
Confidence: 99.98%
----------------------------------------------------------------------

Complaint:
Ye service bilkul bakwas hai.
Prediction: NEGATIVE
Confidence: 99.99%
----------------------------------------------------------------------

Complaint:
Main is service se bohat khush hun.
Prediction: NEGATIVE
Confidence: 99.98%
----------------------------------------------------------------------

Complaint:
Teen din ho gaye hain lekin meri complaint solve nahi hui.
Prediction: NEGATIVE
Confidence: 99.98%
----------------------------------------------------------------